# Create the Labels

## Objective

Create the target label that will be used by the machine learning model.

This notebook reads the ML table artifact produced by the previous notebook, defines the target based on the available order data, creates the labels, and validates the resulting labeled dataset.

## Input

- `artifacts/MlTable.parquet`

## Steps

1. Load the ML table artifact.
2. Define the target variable.
3. Create the labels.
4. Check the label distribution.
5. Validate the labeled dataset.
6. Save the labeled table as an artifact for the next notebook.

## Output

- Labeled ML table artifact

In [ ]:
#Load the ML table artifact.
import pandas as pd
MlTable = pd.read_parquet("../artifacts/ml_table.parquet")
print("Artifact loaded successfully! Shape:", MlTable.shape)

Artifact loaded successfully! Shape: (99441, 40)


In [ ]:
#Define the target variable.
MlTable["order_delivered_customer_date"] = pd.to_datetime(
    MlTable["order_delivered_customer_date"]
)
MlTable["order_estimated_delivery_date"] = pd.to_datetime(
    MlTable["order_estimated_delivery_date"]
)

In [9]:
MlTable["is_late"] = (
    MlTable["order_delivered_customer_date"]
    > MlTable["order_estimated_delivery_date"]
).astype(int)



In [15]:
print(
    MlTable[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "is_late",
        ]
    ].head()
)

                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:00   
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:00   
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:00   
3  949d5b44dbf5de918fe9c16f97b45f8a           2017-12-02 00:28:00   
4  ad21c59c0840e6cb83a9ceb5573f8159           2018-02-16 18:17:00   

  order_estimated_delivery_date  is_late  
0                    2017-10-18        0  
1                    2018-08-13        0  
2                    2018-09-04        0  
3                    2017-12-15        0  
4                    2018-02-26        0  


In [ ]:
# Check the label distribution.
# late 7.87%
# on-time 92.13%
MlTable["is_late"].value_counts(normalize=True)



is_late
0    0.92129
1    0.07871
Name: proportion, dtype: float64

In [17]:
# Save the labeled table as an artifact for the next notebook.

from pathlib import Path

output_path = Path("../artifacts/ml_table_labeled.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

MlTable.to_parquet(output_path, index=False)